In [0]:
import pandas as pd
import numpy as np

# Load data from Unity Catalog
df_spark = spark.table("ardemo_classic_dnubtw_catalog.ws_rico_martinez.mlops_churn_training")
df = df_spark.toPandas()

print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nTarget distribution (churn):")
print(df['churn'].value_counts())
print(f"\nSplit column distribution:")
print(df['split'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() > 0 else "No missing values")
print(f"\nNumeric feature statistics:")
df.describe()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Target distribution
df['churn'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['steelblue', 'coral'])
axes[0, 0].set_title('Churn Distribution')
axes[0, 0].set_ylabel('Count')

# 2. Correlation heatmap of numeric features
numeric_cols = ['tenure', 'monthly_charges', 'total_charges', 'num_optional_services']
df_numeric = df[numeric_cols].copy()
df_numeric['churn_binary'] = (df['churn'] == 'Yes').astype(int)
corr = df_numeric.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[0, 1], fmt='.2f')
axes[0, 1].set_title('Feature Correlation Matrix')

# 3. Monthly charges by churn
sns.boxplot(data=df, x='churn', y='monthly_charges', ax=axes[1, 0], palette=['steelblue', 'coral'])
axes[1, 0].set_title('Monthly Charges by Churn')

# 4. Tenure by churn
sns.boxplot(data=df, x='churn', y='tenure', ax=axes[1, 1], palette=['steelblue', 'coral'])
axes[1, 1].set_title('Tenure by Churn')

plt.tight_layout()
plt.show()

In [0]:
# Feature type summary
feature_info = []
for col in df.columns:
    if col in ['customer_id', 'churn', 'split']:
        continue
    if df[col].dtype in ['float64', 'int64']:
        ftype = 'numeric (float)' if df[col].dtype == 'float64' else 'numeric (integer)'
    else:
        ftype = f'categorical (nominal, {df[col].nunique()} unique)'
    feature_info.append({'Feature': col, 'Type': ftype})

feature_df = pd.DataFrame(feature_info)
display(feature_df)

In [0]:
%pip install category_encoders lightgbm -q
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from lightgbm import LGBMClassifier

# Define features and target
exclude_cols = ['customer_id', 'churn', 'split']
feature_cols = [c for c in df.columns if c not in exclude_cols]

categorical_cols = [c for c in feature_cols if df[c].dtype == 'object']
numeric_cols = ['tenure', 'monthly_charges', 'total_charges', 'num_optional_services']

# Use pre-defined split
train_df = df[df['split'] == 'train'].copy()
test_df = df[df['split'] == 'test'].copy()

X_train = train_df[feature_cols]
y_train = (train_df['churn'] == 'Yes').astype(int)
X_test = test_df[feature_cols]
y_test = (test_df['churn'] == 'Yes').astype(int)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

# Build preprocessing + model pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', ce.OrdinalEncoder(), categorical_cols)
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        verbose=-1
    ))
])

# Train with MLflow tracking
mlflow.set_experiment(f"/Users/rico.martinez@databricks.com/churn_classification_experiment")

with mlflow.start_run(run_name="lgbm_churn_classifier") as run:
    pipeline.fit(X_train, y_train)
    
    # Predictions
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log parameters and metrics
    mlflow.log_params({
        "model_type": "LGBMClassifier",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 6,
        "n_features": len(feature_cols),
        "train_size": X_train.shape[0],
        "test_size": X_test.shape[0]
    })
    mlflow.log_metrics({
        "accuracy": accuracy,
        "f1_score": f1,
        "roc_auc": roc_auc
    })
    
    # Log model with signature
    from mlflow.models.signature import infer_signature
    signature = infer_signature(X_train, y_pred)
    mlflow.sklearn.log_model(pipeline, "model", signature=signature, input_example=X_train.iloc[:3])
    
    print(f"\n{'='*50}")
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [0]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve, auc

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC-AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)
axes[2].plot(recall, precision, color='coral', lw=2, label=f'PR-AUC = {pr_auc:.3f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.show()

# Log plots to MLflow
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metric("pr_auc", pr_auc)
    fig.savefig("/tmp/evaluation_plots.png", dpi=100, bbox_inches='tight')
    mlflow.log_artifact("/tmp/evaluation_plots.png")

print(f"\nPR-AUC: {pr_auc:.4f}")